# 05 — EDA: pLLM zero-shot scores (TEM-1 / Firnberg)

The exploratory step before the `06` zero-shot benchmark. It follows the numbered-pipeline
structure of `01_EDA` (self-contained, reads from disk via `paths.py`, class-separation +
by-position + association-tests), and folds in the deeper per-feature diagnostics from the
project's annotated `ML EDA Notebook` — data-quality checks, per-column distributions and skew,
boxplots, and outlier handling — applied here to the ESM score columns.

**Feature under study:** the zero-shot ESM scores, one column per `{model}__{scheme}` (ESM ladder ×
masked-marginal / wildtype-marginal). Each is a surprisal difference log P(mut) − log P(wt) at the
mutated site. Equivalently this is **surprisal(wt) − surprisal(mut)** — the wild-type-referenced form of the ESM surprisal used in the earlier `ML EDA Notebook` (subtracting the WT surprisal cancels each position's baseline intolerance, isolating the effect of the specific substitution). **Run the Colab twin `05a_pllm_zeroshot_feature_extraction_colab.ipynb` first** and drop its
parquet into `data/features/plm_masked_marginal/`.

In [ ]:
# self-contained: resolve project root via .projectroot, read from disk (never in-memory state)
import sys
from pathlib import Path
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'.projectroot').exists())
sys.path.insert(0, str(root)); from paths import *

import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from sklearn.metrics import roc_auc_score

AA = list("ACDEFGHIKLMNPQRSTVWY")
MODEL_ORDER = ["esm1b","esm1v","esm2_150m","esm2_650m","esm2_3b","esmc_300m","esmc_600m"]
MODEL_LABEL = {"esm1b":"ESM-1b 650M","esm1v":"ESM-1v 650M","esm2_150m":"ESM-2 150M",
               "esm2_650m":"ESM-2 650M","esm2_3b":"ESM-2 3B","esmc_300m":"ESM-C 300M",
               "esmc_600m":"ESM-C 600M"}
SCHEMES=["masked_marginal","wildtype_marginal"]; PRIMARY_SCHEME="masked_marginal"
# project palette: blues, greens, purples, dark pinks only
MODEL_COLORS = {"esm1b":"#2c6fbb","esm1v":"#1f4e8c","esm2_150m":"#57a773","esm2_650m":"#2e8b57",
                "esm2_3b":"#1d6b3f","esmc_300m":"#7a4fa3","esmc_600m":"#b03060"}
PAL={"blue":"#2c6fbb","green":"#2e8b57","purple":"#7a4fa3","pink":"#b03060","grey":"#9aa0a6"}
sns.set_theme(style="whitegrid")

NBNAME="05_EDA_pllm_zeroshot"
FIGDIR=FIGURES/NBNAME; FIGDIR.mkdir(parents=True, exist_ok=True); TABLES.mkdir(parents=True, exist_ok=True)
print("root:", root, "| fig dir:", FIGDIR.relative_to(BASE_DIR))


## 1. Load and validate — trust, but verify the Colab output

In [ ]:
df = pd.read_parquet(PROCESSED/"traditional_ml_aa_identity"/"modeling_dataset.parquet")
SCORES_PATH = FEATURES_PLM_MM/"pllm_zeroshot_scores.parquet"
assert SCORES_PATH.exists(), (
    f"scores not found at {SCORES_PATH}\nRun 05a_pllm_zeroshot_feature_extraction_colab.ipynb first "
    "and drop pllm_zeroshot_scores.parquet into data/features/plm_masked_marginal/.")
scores = pd.read_parquet(SCORES_PATH)
# boundary checks (CLAUDE.md): the score file crossed from an external GPU run, so verify it
assert df.seq_id.is_unique and scores.seq_id.is_unique, "duplicate seq_id"
assert set(scores.seq_id)==set(df.seq_id) and len(scores)==len(df), "score/label seq_id mismatch"

data = df.merge(scores, on="seq_id", how="inner", validate="1:1").sort_values(["position","mut_aa"]).reset_index(drop=True)
label_col="DMS_score_bin"; score_col="DMS_score"
y = data[label_col].values.astype(int)
present=[(m,s) for m in MODEL_ORDER for s in SCHEMES if f"{m}__{s}" in data.columns]
cols=[f"{m}__{s}" for m,s in present]
assert cols, "no {model}__{scheme} columns present"
print(f"variants={len(data)} | class balance={np.bincount(y).tolist()} | score cols={len(cols)}")
print("present:", cols)


## 2. Data quality — trust, but verify (maybe don't even trust, just verify)

Before any interpretation, the same standing checks the project's `ML EDA Notebook` runs on every
feature table: are there missing values, do the numbers make physical sense, and is anything
constant or degenerate. A zero-shot score should be finite everywhere (every position was scored)
and mostly negative (a substitution is usually less likely than the wild-type residue).

In [ ]:
# missingness — a zero-shot score must exist for every variant; a NaN means a model/position
# silently failed on Colab and would poison the benchmark.
miss = pd.DataFrame({"n_missing": data[cols].isna().sum(),
                     "pct_missing": (data[cols].isna().mean()*100).round(2)})
print("columns with any missing value:")
print(miss.query("n_missing > 0") if miss.n_missing.sum() else "  none")
assert data[cols].isna().sum().sum()==0, "NaN scores present — re-check the Colab run"
# physical sense: describe + fraction negative + any degenerate (zero-variance) column
desc = data[cols].describe().T[["mean","std","min","25%","50%","75%","max"]].round(3)
desc["frac_neg"] = (data[cols] < 0).mean().round(3).values
display(desc)
degenerate=[c for c in cols if data[c].std()==0]
assert not degenerate, f"zero-variance score columns: {degenerate}"
print("all scores finite, non-degenerate;", f"{(data[cols]<0).mean().mean():.0%} of scores are negative on average")


## 3. Distribution of every score column, and skew

Histogram per score column, then the skew of each. Surprisal-style scores typically carry a long
left tail — most substitutions are mildly disfavored, a few are strongly forbidden — so a large
negative skew is expected and is itself the signal (the forbidden tail is where non-functional
variants concentrate).

In [ ]:
ncol=4; nrow=int(np.ceil(len(cols)/ncol))
fig,axes=plt.subplots(nrow,ncol,figsize=(3.4*ncol,2.5*nrow)); axes=np.atleast_1d(axes).ravel()
for ax,c in zip(axes,cols):
    m=c.rsplit("__",1)[0]
    sns.histplot(data[c],bins=30,ax=ax,color=MODEL_COLORS.get(m,PAL["blue"]))
    ax.set_title(c,fontsize=8); ax.set_xlabel(""); ax.set_ylabel("")
for ax in axes[len(cols):]: ax.axis("off")
fig.suptitle("Distribution of every zero-shot score column",y=1.01)
fig.tight_layout(); fig.savefig(FIGDIR/"score_histograms.pdf",bbox_inches="tight"); fig.savefig(FIGDIR/"score_histograms.png",bbox_inches="tight", dpi=200); plt.show()
display(data[cols].skew().round(2).sort_values().to_frame("skew"))
print("negative skew = long forbidden-substitution tail, as expected for surprisal scores")


## 4. Boxplots and outliers

Boxplots summarize spread and flag the tails at a glance. For a surprisal score the extreme-low
outliers are not errors to clip — they are the strongly-forbidden substitutions, exactly the
variants most likely to be non-functional. We quantify the low tail with the IQR rule but keep it,
noting how enriched it is for non-functional variants.

In [ ]:
fig,ax=plt.subplots(figsize=(12,4.6))
box=data[cols].copy(); box.columns=[c.replace("__","\n") for c in cols]
sns.boxplot(data=box,ax=ax,color=PAL["grey"],fliersize=1.5)
ax.set_ylabel("score"); ax.set_title("Score spread per model x scheme")
plt.xticks(rotation=0,fontsize=7)
fig.tight_layout(); fig.savefig(FIGDIR/"score_boxplots.pdf",bbox_inches="tight"); fig.savefig(FIGDIR/"score_boxplots.png",bbox_inches="tight", dpi=200); plt.show()

# IQR low-tail enrichment for the primary-scheme best-separating model (defined next section too,
# but computed inline here on the highest-AUC masked-marginal score)
prim_cols=[f"{m}__{PRIMARY_SCHEME}" for m in MODEL_ORDER if f"{m}__{PRIMARY_SCHEME}" in cols]
aucs={c:roc_auc_score(y,data[c]) for c in prim_cols}
lead=max(aucs,key=aucs.get)
q1,q3=data[lead].quantile([0.25,0.75]); iqr=q3-q1; lo=q1-1.5*iqr
tail=data[lead]<lo
print(f"{lead}: IQR low-tail cutoff = {lo:.2f}; {tail.sum()} variants below it")
if tail.sum():
    print(f"  non-functional rate in that tail: {1-y[tail.values].mean():.2%} "
          f"(vs {1-y.mean():.2%} overall) -> forbidden-substitution tail is enriched for knockouts")


## 5. Score distributions by functional class

The central EDA question: does the score separate functional from non-functional variants?
Functional variants should sit at higher (less negative) scores. Shown across the ladder for the
primary (masked-marginal) scheme.

In [ ]:
prim=[m for m in MODEL_ORDER if f"{m}__{PRIMARY_SCHEME}" in cols]
n=len(prim); ncol=3; nrow=int(np.ceil(n/ncol))
fig,axes=plt.subplots(nrow,ncol,figsize=(5*ncol,3.4*nrow)); axes=np.atleast_1d(axes).ravel()
for ax,m in zip(axes,prim):
    s=data[f"{m}__{PRIMARY_SCHEME}"]
    sns.kdeplot(s[y==0],ax=ax,color=PAL["pink"],fill=True,alpha=.35,label="non-functional")
    sns.kdeplot(s[y==1],ax=ax,color=PAL["green"],fill=True,alpha=.35,label="functional")
    ax.set_title(MODEL_LABEL[m]); ax.set_xlabel("masked-marginal score"); ax.legend(fontsize=7,frameon=False)
for ax in axes[n:]: ax.set_visible(False)
fig.suptitle("Zero-shot score distribution by class (masked-marginal)",y=1.01)
fig.tight_layout(); fig.savefig(FIGDIR/"score_distributions.pdf",bbox_inches="tight"); fig.savefig(FIGDIR/"score_distributions.png",bbox_inches="tight", dpi=200); plt.show()


## 6. Separation strength per model × scheme

Two direct read-outs of single-score separability: pooled ROC-AUC (rank separation) and the
standardized mean difference (Cohen's d) between functional and non-functional scores.

In [ ]:
def cohens_d(a,b):
    na,nb=len(a),len(b); va,vb=a.var(ddof=1),b.var(ddof=1)
    sp=np.sqrt(((na-1)*va+(nb-1)*vb)/(na+nb-2))
    return (a.mean()-b.mean())/sp if sp>0 else 0.0
sep_rows=[]
for m,s in present:
    v=data[f"{m}__{s}"].values
    sep_rows.append(dict(model=m,scheme=s,roc_auc=round(roc_auc_score(y,v),4),
                         cohens_d=round(cohens_d(v[y==1],v[y==0]),3)))
sep=pd.DataFrame(sep_rows).sort_values("roc_auc",ascending=False).reset_index(drop=True)
display(sep)
sp=sep[sep.scheme==PRIMARY_SCHEME].set_index("model").reindex(prim)
fig,ax=plt.subplots(figsize=(8,4.4))
ax.bar([MODEL_LABEL[m] for m in prim], sp.roc_auc, color=[MODEL_COLORS[m] for m in prim],
       edgecolor="black", linewidth=.5)
ax.axhline(0.5,ls="--",color=PAL["grey"],lw=1)
ax.set_ylabel("pooled ROC-AUC"); ax.set_ylim(0.4,1.0)
ax.set_title("Single-score separation by model (masked-marginal)")
plt.xticks(rotation=30,ha="right")
fig.tight_layout(); fig.savefig(FIGDIR/"separation_by_model.pdf",bbox_inches="tight"); fig.savefig(FIGDIR/"separation_by_model.png",bbox_inches="tight", dpi=200); plt.show()
best_model=sep[sep.scheme==PRIMARY_SCHEME].iloc[0].model; col=f"{best_model}__{PRIMARY_SCHEME}"


## 7. Score vs position — where along the sequence is the signal?

`01_EDA` found the functional rate has regional structure (motivating the contiguous split). Does
the zero-shot score track it? Rolling-mean score by position for the best model, with the
functional rate underneath, and a direct check at the catalytic residues (Ambler S70, K73, S130,
E166) — surprisal should flag those constrained sites.

In [ ]:
by_pos=data.groupby("position").apply(lambda g: pd.Series({
    "mean_score":g[col].mean(), "func_rate":g[label_col].mean()}))
fig,ax=plt.subplots(2,1,figsize=(13,6),sharex=True)
ax[0].plot(by_pos.index, by_pos.mean_score.rolling(10,center=True,min_periods=3).mean(),
           color=PAL["purple"],lw=1.6)
ax[0].axhline(data[col].mean(),color=PAL["grey"],ls="--",lw=1)
ax[0].set_ylabel("mean score\n(10-res rolling)")
ax[0].set_title(f"Zero-shot score along the sequence — {MODEL_LABEL[best_model]} (masked-marginal)")
ax[1].scatter(by_pos.index, by_pos.func_rate, s=12, color=PAL["green"], alpha=.7)
ax[1].axhline(data[label_col].mean(),color=PAL["pink"],ls="--",lw=1)
ax[1].set_ylabel("P(functional)"); ax[1].set_xlabel("residue position")
ax[1].set_title("Functional rate by position (for reference)")
fig.tight_layout(); fig.savefig(FIGDIR/"score_by_position.pdf",bbox_inches="tight"); fig.savefig(FIGDIR/"score_by_position.png",bbox_inches="tight", dpi=200); plt.show()
cat=[70,73,130,166]; cat_present=[p for p in cat if p in by_pos.index]
print("overall mean score:", round(data[col].mean(),3))
for p in cat_present:
    print(f"  catalytic pos {p}: mean score {by_pos.loc[p,'mean_score']:.3f}, func rate {by_pos.loc[p,'func_rate']:.2f}")


## 8. Agreement with the continuous DMS score

The label binarizes a continuous DMS fitness. A good zero-shot scorer should track the underlying
`DMS_score` too — Spearman rank correlation is the standard zero-shot variant-effect metric (Meier
2021). Scatter plus a hexbin (which shows density where ~4,800 points overplot).

In [ ]:
corr_rows=[]
for m,s in present:
    v=data[f"{m}__{s}"].values
    rho,_=stats.spearmanr(v, data[score_col].values)
    pear=np.corrcoef(v, data[score_col].values)[0,1]
    corr_rows.append(dict(model=m,scheme=s,spearman_dms=round(rho,4),pearson_dms=round(pear,4)))
dms_corr=pd.DataFrame(corr_rows).sort_values("spearman_dms",ascending=False).reset_index(drop=True)
display(dms_corr)
fig,ax=plt.subplots(1,2,figsize=(12,5))
ax[0].scatter(data[col], data[score_col], s=6, alpha=.2, color=PAL["blue"])
rho,_=stats.spearmanr(data[col], data[score_col])
ax[0].set_xlabel(f"{MODEL_LABEL[best_model]} masked-marginal score"); ax[0].set_ylabel("DMS score")
ax[0].set_title(f"scatter (Spearman {rho:.3f})")
hb=ax[1].hexbin(data[col], data[score_col], gridsize=40, cmap="BuPu", mincnt=1)
ax[1].set_xlabel(f"{MODEL_LABEL[best_model]} masked-marginal score"); ax[1].set_ylabel("DMS score")
ax[1].set_title("hexbin density"); fig.colorbar(hb,ax=ax[1],label="count")
fig.suptitle("Zero-shot score vs DMS fitness",y=1.02)
fig.tight_layout(); fig.savefig(FIGDIR/"score_vs_dms.pdf",bbox_inches="tight"); fig.savefig(FIGDIR/"score_vs_dms.png",bbox_inches="tight", dpi=200); plt.show()


## 9. Cross-model and cross-scheme agreement

Do the models see the same thing? High agreement means the ladder measures one shared constraint
signal (scale mostly rescales it); low agreement would mean the models disagree about which
variants are disruptive.

In [ ]:
prim_cols=[f"{m}__{PRIMARY_SCHEME}" for m in prim]
cm=data[prim_cols].corr(method="spearman")
cm.index=[MODEL_LABEL[m] for m in prim]; cm.columns=[MODEL_LABEL[m] for m in prim]
fig,ax=plt.subplots(figsize=(6.5,5.5))
sns.heatmap(cm,annot=True,fmt=".2f",cmap="BuPu",vmin=0,vmax=1,square=True,ax=ax,
            cbar_kws={"label":"Spearman"})
ax.set_title("Cross-model score agreement (masked-marginal)")
fig.tight_layout(); fig.savefig(FIGDIR/"cross_model_agreement.pdf",bbox_inches="tight"); fig.savefig(FIGDIR/"cross_model_agreement.png",bbox_inches="tight", dpi=200); plt.show()
if all(f"{m}__wildtype_marginal" in cols for m in prim):
    print("masked vs wildtype-marginal Spearman per model:")
    for m in prim:
        r,_=stats.spearmanr(data[f"{m}__masked_marginal"], data[f"{m}__wildtype_marginal"])
        print(f"  {MODEL_LABEL[m]}: {r:.3f}")


## 10. Association tests (mirrors `01_EDA_association_tests.csv`)

In [ ]:
rows=[]
for m,s in present:
    v=data[f"{m}__{s}"].values
    pb,ppb=stats.pointbiserialr(y, v)
    u,pu=stats.mannwhitneyu(v[y==1], v[y==0], alternative="two-sided")
    rows.append({"test":f"score {m}__{s}","point_biserial_r":round(pb,4),
                 "mannwhitney_U":int(u),"auc_equiv":round(roc_auc_score(y,v),4),"p_value":pu})
assoc=pd.DataFrame(rows).sort_values("auc_equiv",ascending=False).reset_index(drop=True)
assoc["p_value"]=assoc["p_value"].map(lambda x: f"{x:.2e}")
assoc.to_csv(TABLES/f"{NBNAME}_association_tests.csv", index=False)
sep.to_csv(TABLES/f"{NBNAME}_separation.csv", index=False)
dms_corr.to_csv(TABLES/f"{NBNAME}_dms_correlation.csv", index=False)
display(assoc)
print("saved 3 tables + figures:")
for f in sorted(FIGDIR.glob("*.pdf")): print("  fig:", f.name)


## 11. Summary

What to carry into the `06` benchmark:

- **Data is clean** — every variant scored, no NaN, no degenerate column; scores are
  predominantly negative with the long forbidden-substitution tail expected of surprisal.
- **Signal is present and directional** — functional variants sit at higher (less negative) scores;
  the separation figure and the AUC-equivalent in the association table quantify how strong per
  model, and the low-tail enrichment shows the forbidden tail concentrates non-functional variants.
- **Best model / scheme** — read off `..._separation.csv`; the ladder shows whether scale
  (1b → 1v → 2 → C) buys separation, and masked- vs wildtype-marginal shows whether the cheaper
  scheme loses anything (D031).
- **Positional read-out** — the score-by-position figure shows whether surprisal flags the
  constrained regions and catalytic residues (S70, K73, S130, E166).

**Caveat (D039).** Surprisal is stability-weighted, so a catalytic-but-stable knockout can score as
tolerable. Watch for active-site positions where the score looks benign but the functional rate is
low — where a sequence-only signal is expected to be blind, and where the structural features and
wet-lab panel do the checking.